# Qwen3.5-4B BF16/FP32 merge and local Colab evaluation

This notebook merges the W&B LoRA adapter into a local mixed-precision checkpoint using Qwen3.5's required BF16 model dtype and FP32 recurrent state tensors, then downloads immutable, pre-sliced evaluation prompts derived from the real `sft/test.parquet` split. Each example contains only that section's gold source spans, which sharply reduces prompt length and KV-cache pressure. It continually backs up flat result rows to one Parquet file in the local Colab runtime and uploads the exact file to W&B as a versioned dataset artifact.

The workflow has two phases: merge and verify, then restart only the Python kernel to free all merge-phase A100 VRAM before evaluation. The merged files remain under `/content` across that kernel restart. Nothing is mounted from or saved to Google Drive. There is no atomic-record, JSONL-progress, hashing, or resume harness. A disconnected/deleted Colab runtime still loses its local files, so download the Parquet file before ending the session.

## 1. Install dependencies and restart once

Run this cell once. Package installation can replace NumPy or other compiled extensions that Colab already loaded, so the cell intentionally restarts the Python kernel after installation. It also removes Colab's optional preinstalled TorchAudio because vLLM may upgrade PyTorch to a different CUDA build and this text-only notebook never uses audio. When Colab reconnects, continue at **2. Configuration and local-Colab paths**; do not rerun this install cell unless the runtime itself was recreated.

In [ ]:
%pip install -q -U uv
!uv pip install -q -U unsloth 'wandb>=0.21.0' 'weave>=0.52.0' 'safetensors>=0.5.0' 'pandas>=2.2.0' 'pyarrow>=17.0.0'
!uv pip install -q -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly
%pip uninstall -q -y torchaudio

print('Dependencies installed. Restarting the Python kernel so compiled packages load consistently.')
print('After reconnecting, continue at Section 2; do not rerun this install cell.')
from IPython import get_ipython
install_kernel = getattr(get_ipython(), 'kernel', None)
if install_kernel is None:
    raise RuntimeError('No active IPython kernel. Restart the Python kernel manually before importing Unsloth.')
install_kernel.do_shutdown(restart=True)

## 2. Configuration and local-Colab paths

Start here after the dependency-install restart. This guarantees that NumPy and other compiled packages in memory match the versions installed on disk.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import time
from datetime import datetime, timedelta
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import wandb
import weave
from safetensors import safe_open
from tqdm.auto import tqdm
from weave import EvaluationLogger

BASE_MODEL = 'Qwen/Qwen3.5-4B'
MERGE_MAX_SEQUENCE_LENGTH = 49_152
MODEL_NATIVE_MAX_LENGTH = 262_144
MAX_GENERATION_TOKENS_CAP = 131_072
MIN_GENERATION_TOKENS = 256
THINKING_TOKEN_RESERVE = 1_024
OUTPUT_TOKEN_HEADROOM = 1.15
CONTEXT_ALIGNMENT = 256
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 1.5
REPETITION_PENALTY = 1.0
ENABLE_THINKING = False
SEED = 3407
REQUESTED_BATCH_SIZE = 128
ENGINE_OVERHEAD_RESERVE_GIB = 8.0
GPU_MEMORY_UTILIZATION = 0.95
CACHE_PACKING_FRACTION = 0.90
TEST_LIMIT: int | None = None

DATASET_REPO = 'Haeryz/putusan-structured-extraction'
DATASET_CONFIG = 'sft'
DATASET_SPLIT = 'test'
WANDB_ENTITY = 'haeriz42069-universitas-muhammadiyah-malang'
WANDB_PROJECT = 'Sinergi-training'
ADAPTER_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-lora:v0'
EVAL_INPUT_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-sliced-section-eval-inputs-no-thinking:v0'

IN_COLAB = Path('/content').is_dir() and 'COLAB_RELEASE_TAG' in os.environ
LOCAL_ROOT = Path('/content/qwen3-5-4b-evaluation') if IN_COLAB else Path('artifacts/qwen3-5-4b-evaluation')
ADAPTER_ROOT = LOCAL_ROOT / 'adapter-artifact'
MERGED_MODEL_DIR = LOCAL_ROOT / 'merged-fp16'
OUTPUT_PARQUET = LOCAL_ROOT / 'qwen3-5-4b-test-outputs.parquet'

for directory in (LOCAL_ROOT, ADAPTER_ROOT, MERGED_MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

assert REQUESTED_BATCH_SIZE >= 1
assert TEST_LIMIT is None or TEST_LIMIT >= 1
assert 1 <= MIN_GENERATION_TOKENS <= MAX_GENERATION_TOKENS_CAP
assert MAX_GENERATION_TOKENS_CAP < MODEL_NATIVE_MAX_LENGTH
assert THINKING_TOKEN_RESERVE >= 0
assert OUTPUT_TOKEN_HEADROOM >= 1.0
assert 0.0 < CACHE_PACKING_FRACTION <= 1.0
assert 0.0 <= TEMPERATURE <= 2.0
assert 0.0 < TOP_P <= 1.0
assert TOP_K >= 0
assert 0.0 <= MIN_P <= 1.0
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime.'
print(f'Local runtime root: {LOCAL_ROOT}')
print(f'Offline Parquet backup: {OUTPUT_PARQUET}')
print({
    'max_generation_tokens_cap': MAX_GENERATION_TOKENS_CAP, 'temperature': TEMPERATURE,
    'top_p': TOP_P, 'top_k': TOP_K, 'min_p': MIN_P,
    'presence_penalty': PRESENCE_PENALTY,
    'repetition_penalty': REPETITION_PENALTY,
    'enable_thinking': ENABLE_THINKING,
})
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)')

## 3. Authenticate for the merge phase

Colab Secrets may provide `WANDB_API_KEY` and `HF_TOKEN`; Google Drive is never imported or mounted. This phase only authenticates so the adapter can be downloaded. W&B evaluation and Weave tracing are initialized after the kernel restart.

In [ ]:
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def runtime_secret(name: str) -> str | None:
    if value := os.getenv(name):
        return value
    if userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            pass
    return None

wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    raise RuntimeError('Set WANDB_API_KEY in Colab Secrets or the environment.')
wandb.login(key=wandb_key, relogin=True)
hf_token = runtime_secret('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

print('Authenticated. Continue to the merge cell.')

## 4. Download the adapter and merge it locally to verified BF16/FP32

Prerequisite: after the dependency restart, run the code cells in **Section 2** and **Section 3** before this cell. Do not jump directly from installation to the merge.

In [ ]:
required_merge_state = {
    'Path', 'wandb', 'ADAPTER_ARTIFACT', 'ADAPTER_ROOT', 'BASE_MODEL',
    'MERGE_MAX_SEQUENCE_LENGTH', 'MERGED_MODEL_DIR', 'torch', 'gc', 'json',
    'Counter', 'tqdm', 'safe_open', 'wandb_key',
}
missing_merge_state = sorted(name for name in required_merge_state if name not in globals())
if missing_merge_state:
    raise RuntimeError(
        'Merge prerequisites are missing after the kernel restart: '
        f'{missing_merge_state}. Run the Section 2 configuration cell and Section 3 '
        'authentication cell, then rerun this merge cell.'
    )

from unsloth import FastLanguageModel

artifact_root = Path(wandb.Api().artifact(ADAPTER_ARTIFACT).download(root=str(ADAPTER_ROOT)))
adapter_dir = artifact_root / 'adapter'
if not (adapter_dir / 'adapter_config.json').is_file():
    raise FileNotFoundError(f'{ADAPTER_ARTIFACT} has no adapter/adapter_config.json')
adapter_config = json.loads((adapter_dir / 'adapter_config.json').read_text(encoding='utf-8'))
if (adapter_base := adapter_config.get('base_model_name_or_path')) and adapter_base != BASE_MODEL:
    raise RuntimeError(f'Adapter base {adapter_base!r} != {BASE_MODEL!r}')

if not (MERGED_MODEL_DIR / 'config.json').is_file():
    merge_model, merge_tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(adapter_dir), max_seq_length=MERGE_MAX_SEQUENCE_LENGTH,
        dtype=torch.bfloat16, load_in_4bit=True, text_only=True,
    )
    merge_model.save_pretrained_merged(
        str(MERGED_MODEL_DIR), merge_tokenizer, save_method='merged_16bit',
        maximum_memory_usage=0.65, safe_serialization=None,
    )
    del merge_model, merge_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Reusing the merged checkpoint already present in this local runtime.')

dtype_counts: Counter[str] = Counter()
shards = sorted(MERGED_MODEL_DIR.glob('*.safetensors'))
if not shards:
    raise FileNotFoundError('Merged model has no safetensors shards.')
for shard in tqdm(shards, desc='Verifying merged tensor dtypes'):
    with safe_open(shard, framework='pt', device='cpu') as handle:
        for name in handle.keys():
            dtype_counts[str(handle.get_slice(name).get_dtype())] += 1
merged_config = json.loads((MERGED_MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
merged_text_config = merged_config.get('text_config', merged_config)
declared_model_dtype = merged_text_config.get('dtype', merged_text_config.get('torch_dtype'))
if declared_model_dtype not in {None, 'bfloat16'}:
    raise RuntimeError(f'Expected Qwen3.5 dtype=bfloat16 or omitted, found {declared_model_dtype!r}')
declared_ssm_dtype = merged_text_config.get('mamba_ssm_dtype')
if declared_ssm_dtype not in {None, 'float32'}:
    raise RuntimeError(
        f'Expected Qwen3.5 mamba_ssm_dtype=float32 or omitted, found {declared_ssm_dtype!r}'
    )
floating = {name for name in dtype_counts if name in {'F16', 'BF16', 'F32', 'F64'}}
if 'BF16' not in floating or not floating <= {'BF16', 'F32'}:
    raise RuntimeError(
        f'Expected BF16 weights with optional FP32 recurrent tensors, found {sorted(floating)}'
    )
print(f'BF16/FP32 verification passed: {dict(dtype_counts)}')

## 5. Restart the kernel after the merge

The verified merged checkpoint is already stored under `/content/qwen3-5-4b-evaluation/merged-fp16`, which survives a Colab **kernel/session restart**. Run the next cell once. It restarts only the Python kernel so every Unsloth, Transformers, PyTorch, CUDA-context, and merged-model allocation is released from the A100. Do **not** disconnect, delete, or factory-reset the Colab runtime.

When Colab reconnects, continue directly at **Phase 2 — fresh-kernel evaluation bootstrap**. Do not rerun the merge cell.

In [ ]:
if not (MERGED_MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError('Merged checkpoint config is missing; do not restart yet.')
if not any(MERGED_MODEL_DIR.glob('*.safetensors')):
    raise FileNotFoundError('Merged checkpoint shards are missing; do not restart yet.')
print(f'Checkpoint safely stored at {MERGED_MODEL_DIR.resolve()}')
print('Restarting only the Python kernel to release all merge-phase VRAM...')
from IPython import get_ipython
kernel = getattr(get_ipython(), 'kernel', None)
if kernel is None:
    raise RuntimeError('No active IPython kernel. Use Runtime > Restart session manually.')
kernel.do_shutdown(restart=True)

# Phase 2 — fresh-kernel evaluation bootstrap

Start here after the kernel reconnects. This cell is deliberately self-contained: it restores imports, constants, local paths, authentication, W&B, and optional Weave support without importing Unsloth or loading merge-phase model objects. It verifies the saved BF16/FP32 checkpoint and checks that at least 90% of A100 VRAM is free before vLLM starts.

In [ ]:
import hashlib
import json
import os
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import wandb
import weave
from tqdm.auto import tqdm
from weave import EvaluationLogger

BASE_MODEL = 'Qwen/Qwen3.5-4B'
MODEL_NATIVE_MAX_LENGTH = 262_144
MAX_GENERATION_TOKENS_CAP = 131_072
MIN_GENERATION_TOKENS = 256
THINKING_TOKEN_RESERVE = 1_024
OUTPUT_TOKEN_HEADROOM = 1.15
CONTEXT_ALIGNMENT = 256
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 1.5
REPETITION_PENALTY = 1.0
ENABLE_THINKING = False
SEED = 3407
REQUESTED_BATCH_SIZE = 128
MAX_NUM_BATCHED_TOKENS = 8_192
SUBMISSION_CHUNK_SIZE = 512
EVALUATION_TARGET_SECONDS = 3 * 60 * 60
THROUGHPUT_SAFETY_FACTOR = 0.85
ENABLE_WEAVE_TRACING = False
ENGINE_OVERHEAD_RESERVE_GIB = 8.0
GPU_MEMORY_UTILIZATION = 0.95
CACHE_PACKING_FRACTION = 0.90
TEST_LIMIT: int | None = None
DATASET_REPO = 'Haeryz/putusan-structured-extraction'
DATASET_CONFIG = 'sft'
DATASET_SPLIT = 'test'
WANDB_ENTITY = 'haeriz42069-universitas-muhammadiyah-malang'
WANDB_PROJECT = 'Sinergi-training'
ADAPTER_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-lora:v0'
EVAL_INPUT_ARTIFACT = f'{WANDB_ENTITY}/{WANDB_PROJECT}/qwen3-5-4b-sliced-section-eval-inputs-no-thinking:v0'
IN_COLAB = Path('/content').is_dir() and 'COLAB_RELEASE_TAG' in os.environ
LOCAL_ROOT = Path('/content/qwen3-5-4b-evaluation') if IN_COLAB else Path('artifacts/qwen3-5-4b-evaluation')
ADAPTER_ROOT = LOCAL_ROOT / 'adapter-artifact'
MERGED_MODEL_DIR = LOCAL_ROOT / 'merged-fp16'
OUTPUT_PARQUET = LOCAL_ROOT / 'qwen3-5-4b-test-outputs.parquet'

assert REQUESTED_BATCH_SIZE >= 1
assert MAX_NUM_BATCHED_TOKENS >= REQUESTED_BATCH_SIZE
assert SUBMISSION_CHUNK_SIZE >= REQUESTED_BATCH_SIZE
assert EVALUATION_TARGET_SECONDS > 0
assert 0 < THROUGHPUT_SAFETY_FACTOR <= 1
assert TEST_LIMIT is None or TEST_LIMIT >= 1
assert 1 <= MIN_GENERATION_TOKENS <= MAX_GENERATION_TOKENS_CAP
assert MAX_GENERATION_TOKENS_CAP < MODEL_NATIVE_MAX_LENGTH
assert THINKING_TOKEN_RESERVE >= 0
assert OUTPUT_TOKEN_HEADROOM >= 1.0
assert 0.0 < CACHE_PACKING_FRACTION <= 1.0
assert 0.0 <= TEMPERATURE <= 2.0
assert 0.0 < TOP_P <= 1.0
assert TOP_K >= 0
assert 0.0 <= MIN_P <= 1.0

if not (MERGED_MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError(f'Merged checkpoint missing at {MERGED_MODEL_DIR}; run Phase 1 first.')
if not any(MERGED_MODEL_DIR.glob('*.safetensors')):
    raise FileNotFoundError(f'No merged safetensors shards at {MERGED_MODEL_DIR}.')

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def runtime_secret(name: str) -> str | None:
    if value := os.getenv(name):
        return value
    if userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            pass
    return None

wandb_key = runtime_secret('WANDB_API_KEY')
if not wandb_key:
    raise RuntimeError('Set WANDB_API_KEY in Colab Secrets or the environment.')
wandb.login(key=wandb_key, relogin=True)
hf_token = runtime_secret('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)

run = wandb.init(
    entity=WANDB_ENTITY, project=WANDB_PROJECT,
    name='qwen3-5-4b-bf16-test-evaluation', job_type='model-evaluation',
    config={
        'base_model': BASE_MODEL, 'adapter_artifact': ADAPTER_ARTIFACT,
        'dataset': EVAL_INPUT_ARTIFACT,
        'source_dataset': f'{DATASET_REPO}/{DATASET_CONFIG}/{DATASET_SPLIT}',
        'evaluation_unit': 'target_json section with sliced gold source spans',
        'model_native_max_length': MODEL_NATIVE_MAX_LENGTH,
        'max_generation_tokens_cap': MAX_GENERATION_TOKENS_CAP,
        'thinking_token_reserve': THINKING_TOKEN_RESERVE,
        'output_token_headroom': OUTPUT_TOKEN_HEADROOM,
        'temperature': TEMPERATURE,
        'top_p': TOP_P, 'top_k': TOP_K, 'min_p': MIN_P,
        'presence_penalty': PRESENCE_PENALTY,
        'repetition_penalty': REPETITION_PENALTY,
        'enable_thinking': ENABLE_THINKING,
        'enable_weave_tracing': ENABLE_WEAVE_TRACING,
        'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
        'cache_packing_fraction': CACHE_PACKING_FRACTION,
        'max_live_sequences': REQUESTED_BATCH_SIZE,
        'max_num_batched_tokens': MAX_NUM_BATCHED_TOKENS,
        'engine_overhead_reserve_gib': ENGINE_OVERHEAD_RESERVE_GIB,
    },
)
evaluation_logger = None
if ENABLE_WEAVE_TRACING:
    weave.init(f'{WANDB_ENTITY}/{WANDB_PROJECT}')
    evaluation_logger = EvaluationLogger(
        name='qwen3-5-4b-bf16-sliced-section-test-outputs-only',
        model='Qwen3.5-4B merged BF16/FP32',
        dataset=EVAL_INPUT_ARTIFACT,
    )

if not torch.cuda.is_available():
    raise RuntimeError('Select an A100 GPU runtime before evaluation.')
free_bytes, total_bytes = torch.cuda.mem_get_info()
free_ratio = free_bytes / total_bytes
print(f'Fresh GPU VRAM: {free_bytes / 2**30:.2f}/{total_bytes / 2**30:.2f} GiB free ({free_ratio:.1%})')
if free_ratio < 0.90:
    raise RuntimeError('Less than 90% VRAM is free. Restart the kernel again before loading vLLM.')
print(f'Merged checkpoint ready without loading merge weights: {MERGED_MODEL_DIR}')
print(f'W&B run: {run.url}')

## 6. Download ready-to-generate section prompts from W&B

Phase 2 uses the immutable W&B dataset artifact `qwen3-5-4b-sliced-section-eval-inputs-no-thinking:v0`. It contains all 9,176 section-sliced examples rendered with Qwen3.5's official `enable_thinking=False` chat template, exact prompt/gold token counts, direct-generation allowances, and sequence budgets. This removes about 9.4 million unnecessary thinking-reserve tokens while preserving every test example. The cell downloads one 34.3 MiB Parquet file plus its summary JSON, validates unique row IDs and every prompt SHA-256, and optionally applies `TEST_LIMIT`. It performs no tokenization on the A100 runtime.

In [ ]:
PRECOMPUTED_INPUT_ROOT = LOCAL_ROOT / 'precomputed-eval-inputs'
precomputed_artifact = run.use_artifact(EVAL_INPUT_ARTIFACT)
precomputed_dir = Path(precomputed_artifact.download(root=str(PRECOMPUTED_INPUT_ROOT)))
PRECOMPUTED_PARQUET = precomputed_dir / 'qwen3-5-4b-sliced-section-eval-no-thinking.parquet'
PRECOMPUTED_SUMMARY = precomputed_dir / 'qwen3-5-4b-sliced-section-eval-no-thinking-summary.json'
if not PRECOMPUTED_PARQUET.is_file() or not PRECOMPUTED_SUMMARY.is_file():
    raise FileNotFoundError(f'Incomplete precomputed artifact at {precomputed_dir}')
precomputed_summary = json.loads(PRECOMPUTED_SUMMARY.read_text(encoding='utf-8'))
precomputed_eval = pd.read_parquet(PRECOMPUTED_PARQUET)
required_columns = {
    'no', 'dataset_id', 'source_row_no', 'parent_id', 'corpus', 'section',
    'source_file', 'source_sha256', 'annotator_model', 'extraction_method',
    'purpose', 'split', 'span_count', 'is_empty', 'sliced_input_chars',
    'sliced_input_tokens', 'question', 'prompt', 'prompt_sha256', 'gold_answer',
    'prompt_tokens_estimate', 'gold_tokens', 'max_new_tokens', 'sequence_token_budget',
}
if missing := required_columns - set(precomputed_eval.columns):
    raise RuntimeError(f'Precomputed artifact is missing columns: {sorted(missing)}')
if precomputed_eval['dataset_id'].duplicated().any():
    duplicates = precomputed_eval.loc[precomputed_eval['dataset_id'].duplicated(), 'dataset_id'].head().tolist()
    raise RuntimeError(f'Duplicate precomputed dataset IDs: {duplicates}')
if len(precomputed_eval) != int(precomputed_summary['section_examples']):
    raise RuntimeError('Precomputed Parquet row count differs from its summary JSON.')
if bool(precomputed_summary['enable_thinking']) != ENABLE_THINKING:
    raise RuntimeError('Artifact thinking mode differs from notebook ENABLE_THINKING.')
calculated_prompt_hashes = precomputed_eval['prompt'].map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
if not calculated_prompt_hashes.equals(precomputed_eval['prompt_sha256']):
    raise RuntimeError('Precomputed prompt SHA-256 validation failed.')
if TEST_LIMIT is not None:
    selected_source_rows = sorted(precomputed_eval['source_row_no'].unique())[:TEST_LIMIT]
    precomputed_eval = precomputed_eval[precomputed_eval['source_row_no'].isin(selected_source_rows)].reset_index(drop=True)
print(
    f'Loaded {len(precomputed_eval):,} ready prompts from {precomputed_artifact.qualified_name}; '
    f'source rows={precomputed_eval["source_row_no"].nunique():,}; '
    f'sliced input tokens p50={precomputed_eval["sliced_input_tokens"].median():,.0f}; '
    f'complete prompt tokens p50={precomputed_eval["prompt_tokens_estimate"].median():,.0f}, '
    f'p95={precomputed_eval["prompt_tokens_estimate"].quantile(0.95):,.0f}, max={precomputed_eval["prompt_tokens_estimate"].max():,}.'
)


## 7. Recalculate A100 memory from artifact token budgets and start vLLM

The verified non-thinking artifact has sliced-source p50 27 and complete-prompt p50 493, p95 4,579, max 84,851. Its largest prompt-plus-direct-generation allowance is 173,766 tokens, giving aligned `max_model_len=173,824`. This cell assigns the A100 cache from actual merged checkpoint bytes and allows up to 128 live vLLM sequences. vLLM performs continuous, memory-aware asynchronous scheduling inside each 512-request submission chunk; it does not reserve 128 maximum-length sequences at once. No tokenizer is loaded in this phase.

### Three-hour optimized target and live estimate

The artifact contains 10,543,896 prompt tokens and 7,135,325 direct-generation allowance tokens: 17,679,221 planned tokens over all 9,176 examples. A three-hour completion requires about 1,637 aggregate prompt-plus-completion tokens/s. Requests are grouped by corpus and section to maximize shared-prefix cache hits, sorted longest-first inside each group to reduce the final long-request tail, and asynchronously scheduled with up to 128 live sequences. After every 512-request chunk the notebook displays rows, percentage, elapsed time, conservative ETA, projected finish clock time, three-hour target remaining, and measured token throughput. The target never aborts generation and never shortens output allowances: every row continues until the complete eval file is written.

| Runtime profile | Full workload | Completion contract |
|---|---:|---:|
| A100 40 GB, BF16 vLLM, up to 128 live sequences | 17.68M planned tokens | optimized for a 3 h target; always continues to the complete file |
| Base M5 16 GB proxy, MLX q4, batch 1 | Same prompts/allowances | about 49.9–75.8 h; no three-hour path |

Apple currently lists the [Mac mini with M4/M4 Pro](https://www.apple.com/mac-mini/specs/), not an M5 Mac mini. The Apple row is therefore a hypothetical base-M5 16 GB proxy using the [base M5 MacBook Air's 153 GB/s unified-memory specification](https://www.apple.com/macbook-air/specs/) and the public [MLX-LM benchmark methodology](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/BENCHMARKS.md). It is not apples-to-apples: this CUDA/vLLM BF16 notebook cannot run on Apple Silicon, and a 16 GB machine would need an MLX quantized model with batch size 1.

In [ ]:
# Colab/Jupyter has already initialized CUDA while sizing the A100. Keep the
# V1 EngineCore in this process instead of spawning an unimportable notebook child.
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
import io
import sys
from contextlib import contextmanager
from transformers import AutoConfig
from vllm import LLM, SamplingParams
import vllm.distributed.parallel_state as vllm_parallel_state

try:
    sys.stdout.fileno()
except (AttributeError, OSError, io.UnsupportedOperation):
    @contextmanager
    def notebook_safe_suppress_stdout():
        yield
    # vLLM's local suppressor requires a real stdout FD, which ipykernel lacks.
    vllm_parallel_state.suppress_stdout = notebook_safe_suppress_stdout

# Unsloth exports the official wrapper-prefixed language_model.* tensors but
# writes only the inner text config. Restore the matching official outer config;
# language_model_only keeps the absent vision tower unallocated.
official_qwen_config = AutoConfig.from_pretrained(BASE_MODEL, token=hf_token)
if official_qwen_config.model_type != 'qwen3_5':
    raise RuntimeError(f'Unexpected base config type: {official_qwen_config.model_type!r}')
official_qwen_config.save_pretrained(MERGED_MODEL_DIR)

work: list[dict[str, Any]] = []
for artifact_row in precomputed_eval.to_dict('records'):
    work.append({
        'no': int(artifact_row['no']), 'row': artifact_row,
        'section': artifact_row['section'], 'question': artifact_row['question'],
        'prompt': artifact_row['prompt'],
        'prompt_tokens_estimate': int(artifact_row['prompt_tokens_estimate']),
        'gold_tokens': int(artifact_row['gold_tokens']),
        'max_new_tokens': int(artifact_row['max_new_tokens']),
        'sequence_token_budget': int(artifact_row['sequence_token_budget']),
    })
prompt_token_counts = precomputed_eval['prompt_tokens_estimate']
gold_token_counts = precomputed_eval['gold_tokens']
sequence_token_budgets = precomputed_eval['sequence_token_budget']
MAX_MODEL_LENGTH = min(MODEL_NATIVE_MAX_LENGTH, ((int(sequence_token_budgets.max()) + CONTEXT_ALIGNMENT - 1) // CONTEXT_ALIGNMENT) * CONTEXT_ALIGNMENT)
if MAX_MODEL_LENGTH < int(sequence_token_budgets.max()):
    raise RuntimeError('Precomputed sequence budget exceeds the configured model context.')

TENSOR_PARALLEL_SIZE = int(os.getenv('SINERGI_TENSOR_PARALLEL_SIZE', str(torch.cuda.device_count())))
if not 1 <= TENSOR_PARALLEL_SIZE <= torch.cuda.device_count():
    raise ValueError(f'Invalid tensor parallel size: {TENSOR_PARALLEL_SIZE}')
gpu_properties = [torch.cuda.get_device_properties(index) for index in range(TENSOR_PARALLEL_SIZE)]
if TENSOR_PARALLEL_SIZE == 1:
    gpu = gpu_properties[0]
    if 'A100' not in gpu.name or not 38 <= gpu.total_memory / 2**30 <= 42:
        raise RuntimeError(
            f'This memory profile requires one A100 40 GB; found {gpu.name} '
            f'with {gpu.total_memory / 2**30:.1f} GiB.'
        )

# Qwen3.5 hybrid cache: BF16 KV for full-attention layers plus one FP32
# Gated DeltaNet recurrent state per live sequence. All figures are per GPU.
saved_config = json.loads((MERGED_MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
text_config = saved_config.get('text_config', saved_config)
layer_types = list(text_config.get('layer_types', []))
num_layers = int(text_config['num_hidden_layers'])
if layer_types:
    full_attention_layers = layer_types.count('full_attention')
    linear_attention_layers = layer_types.count('linear_attention')
else:
    full_attention_interval = int(text_config['full_attention_interval'])
    full_attention_layers = num_layers // full_attention_interval
    linear_attention_layers = num_layers - full_attention_layers
kv_element_bytes = 2
kv_bytes_per_token = (
    full_attention_layers * 2 * int(text_config['num_key_value_heads'])
    * int(text_config['head_dim']) * kv_element_bytes
)
ssm_element_bytes = 4 if text_config.get('mamba_ssm_dtype') == 'float32' else 2
delta_state_bytes_per_sequence = (
    linear_attention_layers * int(text_config['linear_num_value_heads'])
    * int(text_config['linear_key_head_dim'])
    * int(text_config['linear_value_head_dim']) * ssm_element_bytes
)
model_weight_bytes = sum(path.stat().st_size for path in MERGED_MODEL_DIR.glob('*.safetensors'))
per_gpu_weight_bytes = model_weight_bytes / TENSOR_PARALLEL_SIZE
smallest_gpu_bytes = min(props.total_memory for props in gpu_properties)
engine_reserve_bytes = ENGINE_OVERHEAD_RESERVE_GIB * 2**30
cache_budget_per_gpu = (
    smallest_gpu_bytes * GPU_MEMORY_UTILIZATION
    - per_gpu_weight_bytes - engine_reserve_bytes
)
if cache_budget_per_gpu <= 0:
    raise RuntimeError('No KV/state-cache budget remains after weights and engine reserve.')
packing_cache_limit_per_gpu = cache_budget_per_gpu * CACHE_PACKING_FRACTION

def sequence_cache_bytes(item: dict[str, Any]) -> float:
    return (
        item['sequence_token_budget'] * kv_bytes_per_token
        + delta_state_bytes_per_sequence
    ) / TENSOR_PARALLEL_SIZE

for item in work:
    item['estimated_cache_bytes_per_gpu'] = sequence_cache_bytes(item)
    if item['estimated_cache_bytes_per_gpu'] > packing_cache_limit_per_gpu:
        raise RuntimeError(
            f'{item["row"]["dataset_id"]} alone needs '
            f'{item["estimated_cache_bytes_per_gpu"] / 2**30:.2f} GiB cache/GPU, '
            f'above the {packing_cache_limit_per_gpu / 2**30:.2f} GiB packing limit.'
        )

# Submit a deep queue to vLLM so its scheduler, rather than Python, continuously
# admits up to 128 sequences according to the actual live cache footprint.
generation_work = sorted(
    work,
    key=lambda item: (
        item['row']['corpus'], item['section'], -item['sequence_token_budget']
    ),
)
submission_chunks = [
    generation_work[start:start + SUBMISSION_CHUNK_SIZE]
    for start in range(0, len(generation_work), SUBMISSION_CHUNK_SIZE)
]
planned_token_total = sum(
    item['prompt_tokens_estimate'] + item['max_new_tokens'] for item in work
)
target_tokens_per_second = planned_token_total / EVALUATION_TARGET_SECONDS
impossible_native_context_cache_bytes = REQUESTED_BATCH_SIZE * (
    MODEL_NATIVE_MAX_LENGTH * kv_bytes_per_token + delta_state_bytes_per_sequence
) / TENSOR_PARALLEL_SIZE
largest_sequence_cache_bytes = max(item['estimated_cache_bytes_per_gpu'] for item in work)
run.config.update({
    'sliced_examples': len(work), 'max_model_length': MAX_MODEL_LENGTH,
    'prompt_tokens_p50': int(prompt_token_counts.quantile(0.50)),
    'prompt_tokens_p95': int(prompt_token_counts.quantile(0.95)),
    'prompt_tokens_max': int(prompt_token_counts.max()),
    'gold_tokens_max': int(gold_token_counts.max()),
    'sequence_token_budget_max': int(sequence_token_budgets.max()),
    'max_live_sequences': REQUESTED_BATCH_SIZE,
    'submission_chunk_size': SUBMISSION_CHUNK_SIZE,
    'evaluation_target_seconds': EVALUATION_TARGET_SECONDS,
    'planned_token_total': planned_token_total,
    'target_tokens_per_second': target_tokens_per_second,
    'model_weight_gib': model_weight_bytes / 2**30,
    'kv_bytes_per_token': kv_bytes_per_token,
    'delta_state_mib_per_sequence': delta_state_bytes_per_sequence / 2**20,
    'kv_cache_budget_gib_per_gpu': cache_budget_per_gpu / 2**30,
    'packing_cache_limit_gib_per_gpu': packing_cache_limit_per_gpu / 2**30,
    'largest_sequence_cache_gib_per_gpu': largest_sequence_cache_bytes / 2**30,
    'impossible_native_context_max_live_cache_gib_per_gpu': impossible_native_context_cache_bytes / 2**30,
})
print(
    f'A100 memory recalculation per GPU: total={smallest_gpu_bytes / 2**30:.2f} GiB, '
    f'weights={per_gpu_weight_bytes / 2**30:.2f} GiB, '
    f'engine reserve={ENGINE_OVERHEAD_RESERVE_GIB:.2f} GiB, '
    f'cache budget={cache_budget_per_gpu / 2**30:.2f} GiB, '
    f'{CACHE_PACKING_FRACTION:.0%} packing limit='
    f'{packing_cache_limit_per_gpu / 2**30:.2f} GiB.\n'
    f'{REQUESTED_BATCH_SIZE} x {MODEL_NATIVE_MAX_LENGTH:,} theoretical reservation (not allocated)='
    f'{impossible_native_context_cache_bytes / 2**30:.2f} GiB; '
    f'prompt tokens p50={prompt_token_counts.quantile(0.50):,.0f}, '
    f'p95={prompt_token_counts.quantile(0.95):,.0f}, max={prompt_token_counts.max():,}; '
    f'sliced max_model_len={MAX_MODEL_LENGTH:,}; chunks={len(submission_chunks):,}; '
    f'max live sequences={REQUESTED_BATCH_SIZE}; '
    f'largest sequence cache={largest_sequence_cache_bytes / 2**30:.2f} GiB; '
    f'3-hour target={target_tokens_per_second:,.0f} planned tok/s.'
)
vllm_hf_overrides = {'architectures': ['Qwen3_5ForConditionalGeneration']}
engine = LLM(
    model=str(MERGED_MODEL_DIR), tokenizer=str(MERGED_MODEL_DIR), dtype='bfloat16',
    hf_overrides=vllm_hf_overrides,
    language_model_only=True,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE, max_model_len=MAX_MODEL_LENGTH,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    kv_cache_memory_bytes=int(cache_budget_per_gpu),
    max_num_seqs=REQUESTED_BATCH_SIZE, max_num_batched_tokens=MAX_NUM_BATCHED_TOKENS,
    async_scheduling=True,
    enable_chunked_prefill=True,
    enable_prefix_caching=True, trust_remote_code=False,
)

def sampling_for(item: dict[str, Any]) -> SamplingParams:
    return SamplingParams(
        max_tokens=item['max_new_tokens'], temperature=TEMPERATURE,
        top_p=TOP_P, top_k=TOP_K, min_p=MIN_P,
        presence_penalty=PRESENCE_PENALTY, repetition_penalty=REPETITION_PENALTY,
        seed=SEED,
    )
free_after_bytes, total_after_bytes = torch.cuda.mem_get_info()
print(
    f'vLLM ready: max live sequences={REQUESTED_BATCH_SIZE}, tensor_parallel={TENSOR_PARALLEL_SIZE}, '
    f'gpu_memory_utilization={GPU_MEMORY_UTILIZATION:.0%}, '
    f'VRAM free after load={free_after_bytes / 2**30:.2f}/{total_after_bytes / 2**30:.2f} GiB'
)

## 8. Generate, trace LLM outputs, and persist the Parquet dataset

This submits every sliced section example in 512-request chunks while vLLM asynchronously schedules up to 128 live sequences. Qwen thinking is disabled using the official chat template and direct-mode sampling settings. vLLM shows live progress within each chunk; the outer progress bar shows total rows, elapsed time, conservative ETA, projected finish clock time, aggregate token throughput, and whether the three-hour target is on track. The target never raises an exception or truncates generation. Results are sorted back to original dataset order and backed up to Parquet after every chunk, so a complete file is always produced when all rows finish, even if actual runtime exceeds the estimate.

In [ ]:
OUTPUT_COLUMNS = [
    'no', 'corpus', 'section', 'question', 'thinking_trace', 'answer',
    'raw_output', 'gold_answer',
    'dataset_id', 'source_row_no', 'parent_id', 'source_file', 'source_sha256',
    'annotator_model', 'extraction_method', 'purpose', 'split',
    'span_count', 'is_empty', 'sliced_input_chars', 'sliced_input_tokens',
    'finish_reason', 'stop_reason', 'prompt_tokens', 'prompt_tokens_estimate',
    'gold_tokens', 'completion_tokens', 'sequence_token_budget',
    'elapsed_seconds', 'max_new_tokens', 'temperature', 'top_p', 'top_k',
    'min_p', 'presence_penalty', 'repetition_penalty', 'enable_thinking',
    'max_model_length', 'max_live_sequences', 'submission_chunk_size',
    'estimated_sequence_cache_gib',
    'kv_cache_budget_gib', 'packing_cache_limit_gib', 'gpu_memory_utilization',
]

def split_qwen_output(raw_output: str) -> tuple[str | None, str]:
    if '</think>' not in raw_output:
        if ENABLE_THINKING:
            return raw_output.strip(), ''
        return None, raw_output.strip()
    thinking_trace, answer = raw_output.split('</think>', 1)
    thinking_trace = thinking_trace.removeprefix('<think>').strip()
    return thinking_trace, answer.strip()

results: list[dict[str, Any]] = []
processed_tokens = 0
evaluation_started = time.perf_counter()
pd.DataFrame(columns=OUTPUT_COLUMNS).to_parquet(OUTPUT_PARQUET, index=False)
progress = tqdm(total=len(work), desc='A100 eval (3h target)', unit='row', dynamic_ncols=True)

for chunk_index, chunk in enumerate(submission_chunks):
    chunk_started = time.perf_counter()
    chunk_number = chunk_index + 1
    total_chunks = len(submission_chunks)
    print(
        f'Chunk {chunk_number}/{total_chunks}: rows {len(results) + 1:,}-'
        f'{len(results) + len(chunk):,} of {len(work):,}'
    )
    generated = engine.generate(
        [item['prompt'] for item in chunk],
        [sampling_for(item) for item in chunk],
        use_tqdm=True,
    )
    chunk_elapsed = time.perf_counter() - chunk_started
    if len(generated) != len(chunk):
        raise RuntimeError(f'vLLM returned {len(generated)} outputs for {len(chunk)} prompts')
    for item, request_output in zip(chunk, generated, strict=True):
        if not request_output.outputs:
            raise RuntimeError(f'No completion for {item["row"]["dataset_id"]}')
        completion = request_output.outputs[0]
        raw_output = completion.text
        thinking_trace, answer = split_qwen_output(raw_output)
        model_output = {
            'raw_output': raw_output,
            'thinking_trace': thinking_trace,
            'answer': answer,
        }
        if evaluation_logger is not None:
            prediction = evaluation_logger.log_prediction(inputs={}, output=model_output)
            prediction.finish()
        row = item['row']
        results.append({
            'no': item['no'],
            'corpus': row['corpus'],
            'section': item['section'],
            'question': item['question'],
            'thinking_trace': thinking_trace,
            'answer': answer,
            'raw_output': raw_output,
            'gold_answer': row['gold_answer'],
            'dataset_id': row['dataset_id'],
            'source_row_no': row['source_row_no'],
            'parent_id': row['parent_id'],
            'source_file': row['source_file'],
            'source_sha256': row['source_sha256'],
            'annotator_model': row.get('annotator_model'),
            'extraction_method': row.get('extraction_method'),
            'purpose': row.get('purpose'),
            'split': row.get('split'),
            'span_count': row['span_count'],
            'is_empty': row['is_empty'],
            'sliced_input_chars': row['sliced_input_chars'],
            'sliced_input_tokens': row['sliced_input_tokens'],
            'finish_reason': completion.finish_reason,
            'stop_reason': str(completion.stop_reason) if completion.stop_reason is not None else None,
            'prompt_tokens': len(request_output.prompt_token_ids),
            'prompt_tokens_estimate': item['prompt_tokens_estimate'],
            'gold_tokens': item['gold_tokens'],
            'completion_tokens': len(completion.token_ids),
            'sequence_token_budget': item['sequence_token_budget'],
            'elapsed_seconds': chunk_elapsed,
            'max_new_tokens': item['max_new_tokens'],
            'temperature': TEMPERATURE,
            'top_p': TOP_P,
            'top_k': TOP_K,
            'min_p': MIN_P,
            'presence_penalty': PRESENCE_PENALTY,
            'repetition_penalty': REPETITION_PENALTY,
            'enable_thinking': ENABLE_THINKING,
            'max_model_length': MAX_MODEL_LENGTH,
            'max_live_sequences': REQUESTED_BATCH_SIZE,
            'submission_chunk_size': len(chunk),
            'estimated_sequence_cache_gib': item['estimated_cache_bytes_per_gpu'] / 2**30,
            'kv_cache_budget_gib': cache_budget_per_gpu / 2**30,
            'packing_cache_limit_gib': packing_cache_limit_per_gpu / 2**30,
            'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
        })
        processed_tokens += len(request_output.prompt_token_ids) + len(completion.token_ids)
    backup = pd.DataFrame(results, columns=OUTPUT_COLUMNS).sort_values('no')
    backup.to_parquet(OUTPUT_PARQUET, index=False)
    elapsed_total = time.perf_counter() - evaluation_started
    measured_tokens_per_second = processed_tokens / elapsed_total
    conservative_tokens_per_second = measured_tokens_per_second * THROUGHPUT_SAFETY_FACTOR
    remaining = [
        item
        for future_chunk in submission_chunks[chunk_index + 1:]
        for item in future_chunk
    ]
    planned_remaining = sum(
        item['prompt_tokens_estimate'] + item['max_new_tokens'] for item in remaining
    )
    eta_seconds = planned_remaining / conservative_tokens_per_second if remaining else 0.0
    projected_finish = datetime.now() + timedelta(seconds=eta_seconds)
    target_remaining = max(0.0, EVALUATION_TARGET_SECONDS - elapsed_total)
    target_status = 'ON TRACK' if eta_seconds <= target_remaining else 'RUNNING PAST 3H'
    progress.update(len(chunk))
    progress.set_postfix_str(
        f'elapsed={tqdm.format_interval(elapsed_total)}, '
        f'ETA={tqdm.format_interval(eta_seconds)}, finish={projected_finish:%H:%M:%S}, '
        f'3h_target_left={tqdm.format_interval(target_remaining)}, '
        f'rate={conservative_tokens_per_second:,.0f} tok/s, {target_status}'
    )
    print(
        f'Local backup: {len(results):,}/{len(work):,} rows; '
        f'conservative ETA {tqdm.format_interval(eta_seconds)}; '
        f'projected finish {projected_finish:%Y-%m-%d %H:%M:%S}; '
        f'3h target remaining {tqdm.format_interval(target_remaining)}; '
        f'{target_status}. Continuing until the complete eval file is written -> {OUTPUT_PARQUET}'
    )
progress.close()

evaluation_elapsed_seconds = time.perf_counter() - evaluation_started
if evaluation_logger is not None:
    evaluation_logger.log_summary()
run.summary['evaluation_elapsed_seconds'] = evaluation_elapsed_seconds
run.summary['evaluation_rows_completed'] = len(results)
run.summary['evaluation_completion_tokens'] = sum(row['completion_tokens'] for row in results)
run.summary['three_hour_target_met'] = (
    evaluation_elapsed_seconds <= EVALUATION_TARGET_SECONDS and len(results) == len(work)
)
print(
    f'Finished {len(results)} outputs in {evaluation_elapsed_seconds / 3600:.2f} hours '
    f'({evaluation_elapsed_seconds:,.1f} seconds). Local Parquet: {OUTPUT_PARQUET.resolve()}'
)

## 9. Upload the exact Parquet file to W&B cloud

This validates the complete column list, uploads the `.parquet` file as a versioned W&B dataset artifact, waits for the cloud upload to finish, and closes the W&B run. It does not send Parquet rows to Weave. Run it manually after an interrupted generation to upload the latest local batches.

In [ ]:
parquet_data = pd.read_parquet(OUTPUT_PARQUET)
if parquet_data.columns.tolist() != OUTPUT_COLUMNS:
    raise RuntimeError(
        f'Parquet columns differ from OUTPUT_COLUMNS: {parquet_data.columns.tolist()}'
    )
parquet_artifact = wandb.Artifact(
    name='qwen3-5-4b-test-results-parquet',
    type='dataset',
    description='Complete Qwen3.5-4B per-section outputs derived from sft/test.parquet target_json, in the same flat schema as the local Colab backup.',
    metadata={
        'row_count': len(parquet_data),
        'columns': OUTPUT_COLUMNS,
        'dataset_repo': DATASET_REPO,
        'dataset_config': DATASET_CONFIG,
        'dataset_split': DATASET_SPLIT,
        'evaluation_unit': 'one target_json section with sliced gold source spans',
        'source_row_count': int(precomputed_summary['source_rows']),
        'unique_parent_ids': int(precomputed_summary['unique_parent_ids']),
        'precomputed_input_artifact': precomputed_artifact.qualified_name,
        'section_example_count': len(work),
        'evaluation_elapsed_seconds': evaluation_elapsed_seconds,
        'evaluation_elapsed_hours': evaluation_elapsed_seconds / 3600,
        'evaluation_target_seconds': EVALUATION_TARGET_SECONDS,
        'three_hour_target_met': evaluation_elapsed_seconds <= EVALUATION_TARGET_SECONDS,
        'completion_token_count': int(parquet_data['completion_tokens'].sum()),
    },
)
parquet_artifact.add_file(
    local_path=str(OUTPUT_PARQUET),
    name=OUTPUT_PARQUET.name,
)
logged_artifact = run.log_artifact(parquet_artifact, aliases=['latest'])
logged_artifact.wait()
print(f'W&B Parquet artifact: {logged_artifact.name}')
print(f'Cloud rows: {len(parquet_data)}; columns: {len(OUTPUT_COLUMNS)}')
run.finish()

## Inspect and download the same local Parquet file
saved = pd.read_parquet(OUTPUT_PARQUET)
print(f'Saved rows: {len(saved)}')
display(saved[['no', 'corpus', 'section', 'thinking_trace', 'answer']].head(3))

if IN_COLAB:
    from google.colab import files
    files.download(str(OUTPUT_PARQUET))
else:
    print(f'Copy this file before deleting the runtime: {OUTPUT_PARQUET.resolve()}')